# 👄 Visage parlant : version interface web

Même moteur (MuseTalk 1.5), mais avec un **petit site** pour envoyer tes fichiers, écrire un texte et récupérer la vidéo, sans le bouton qui expire.

> ⚠️ **À respecter**
> - N'utilise que des visages et des voix dont tu as **l'accord**.
> - Indique que la vidéo est **générée par IA** quand tu la publies.

## Comment faire
1. **Exécution → Modifier le type d'exécution → T4 GPU → Enregistrer.**
2. **Exécution → Tout exécuter.** L'installation prend 10 à 25 min la première fois.
3. Quand la dernière cellule affiche un lien **public URL** (`...gradio.live`), ouvre-le.
4. Dans le site : photo ou vidéo du visage, audio ou texte, case d'accord, **Animer**.

**Important**
- **Garde l'onglet Colab ouvert** pendant toute l'utilisation. Sur téléphone, si le navigateur met l'onglet en veille, la génération peut s'arrêter.
- Le lien public est **temporaire** : il change à chaque lancement.
- Audio ou texte : 60 s maximum.

In [ ]:
# ÉTAPE 1 : réglages + vérification du GPU
OUTPUT_NAME = "visage-parlant"  #@param {type:"string"}
FPS = 25  #@param {type:"integer"}
import subprocess, os, re
OUTPUT_NAME = re.sub(r"[^A-Za-z0-9._-]+", "_", OUTPUT_NAME.strip()) or "visage-parlant"
FPS = int(FPS) if int(FPS) > 0 else 25
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if r.returncode != 0 or not r.stdout.strip():
    raise SystemExit("❌ Pas de GPU. Menu Exécution > Modifier le type d'exécution > T4 GPU > Enregistrer, puis relance tout.")
print("✅ GPU trouvé :", r.stdout.strip())
for d in ("/content/work", "/content/output", "/content/inputs"): os.makedirs(d, exist_ok=True)

In [ ]:
# ÉTAPE 2 : installation de MuseTalk 1.5 (5 à 8 min la première fois, puis presque instantané)
import subprocess, os, sys, time, glob
MT_LOG = "/content/work/musetalk.log"
PY = "/content/mtenv/bin/python"
MT_DIR = "/content/MuseTalk"

def mt_run(cmd, label):
    print(f"   ▶ {label}")
    with open(MT_LOG, "a") as log:
        log.write(f"\n\n===== {time.strftime('%H:%M:%S')} {label}\n$ {cmd}\n")
    p = subprocess.run(cmd, shell=True, executable="/bin/bash", capture_output=True, text=True)
    out = (p.stdout or "") + (p.stderr or "")
    with open(MT_LOG, "a") as log:
        log.write(out + f"\n[exit code {p.returncode}]\n")
    return p.returncode == 0, out

def mt_env_report():
    if not os.path.exists(PY):
        print("ENV : environnement MuseTalk non installé"); return
    chk = r"""
import sys, importlib
print('python', sys.version.split()[0])
try:
    import torch; print('torch', torch.__version__, '| cuda dispo:', torch.cuda.is_available())
except Exception as e: print('torch ECHEC:', repr(e))
for m in ['numpy','cv2','mmcv','mmengine','mmdet','mmpose','mmpose.apis','mmcv.ops','diffusers','transformers','librosa']:
    try:
        x = importlib.import_module(m); print(f'{m:16s} OK ', getattr(x,'__version__',''))
    except Exception as e: print(f'{m:16s} ECHEC:', repr(e)[:200])
"""
    p = subprocess.run([PY, "-c", chk], capture_output=True, text=True, cwd=MT_DIR if os.path.isdir(MT_DIR) else None)
    print("---- vérification de l'environnement ----"); print(p.stdout, p.stderr[-1500:])

def mt_fail(label, out):
    lines = out.strip().splitlines()[-60:]
    print("\n" + "=" * 70)
    print(f"❌ MuseTalk a échoué à l'étape : {label}")
    print("---- 60 dernières lignes (fais une capture d'écran de ce cadre) ----")
    print("\n".join(lines) if lines else "(aucune sortie)")
    print("=" * 70)
    print("Journal complet :", MT_LOG)
    mt_env_report()
    raise RuntimeError(label)

def mt_install():
    PIP = f"{PY} -m pip install -q"
    steps = []
    if not os.path.isdir(MT_DIR):
        steps.append(("téléchargement de MuseTalk", f"git clone -q https://github.com/TMElyralab/MuseTalk {MT_DIR}"))
    if not os.path.exists(PY):
        steps += [("installation de uv", "pip -q install uv"),
                  ("création de l'environnement Python 3.10", "uv venv -q --seed --python 3.10 /content/mtenv")]
    marker = "/content/mtenv/.musetalk_ok"
    if not os.path.exists(marker):
        steps += [
          ("torch 2.0.1 cu118", f"{PIP} torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118"),
          ("dépendances MuseTalk (versions fixées)", f"{PIP} numpy==1.23.5 diffusers==0.30.2 accelerate==0.28.0 transformers==4.39.2 tokenizers==0.15.2 huggingface_hub==0.30.2 "
             "opencv-python==4.9.0.80 soundfile==0.12.1 librosa==0.11.0 numba==0.59.1 einops==0.8.1 omegaconf==2.3.0 "
             "ffmpeg-python==0.2.0 moviepy==1.0.3 'imageio[ffmpeg]==2.34.0' tqdm requests"),
          ("mmengine", f"{PIP} mmengine==0.10.4"),
          ("mmcv 2.0.1 (version précompilée)", f"{PIP} --only-binary=mmcv mmcv==2.0.1 -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html"),
          ("mmdet 3.1.0", f"{PIP} mmdet==3.1.0"),
          ("mmpose 1.1.0", f"{PIP} json_tricks munkres xtcocotools==1.14.3 'scipy==1.11.4' 'matplotlib==3.8.4' 'pycocotools==2.0.7' 'shapely==2.0.4' terminaltables && {PIP} --no-deps mmpose==1.1.0"),
          ("setuptools<70 (pkg_resources)", f"{PIP} 'setuptools<70'"),
          ("re-fixer numpy/opencv", f"{PIP} --force-reinstall --no-deps numpy==1.23.5 opencv-python==4.9.0.80"),
        ]
    for label, cmd in steps:
        ok, out = mt_run(cmd, label)
        if not ok: mt_fail(label, out)
    ok, out = mt_run(f"{PY} -c 'import pkg_resources' || {PIP} 'setuptools<70'", "vérification pkg_resources")
    if not ok: mt_fail("pkg_resources", out)
    ok, out = mt_run(f"cd {MT_DIR} && {PY} -c \"import pkg_resources,torch,cv2,numpy,mmcv,mmcv.ops,mmdet,mmpose,mmpose.apis; from mmpose.apis import init_model; "
                     "from transformers import WhisperModel; from diffusers import AutoencoderKL; import musetalk.utils.utils; "
                     "print('imports OK | torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| numpy', numpy.__version__)\"",
                     "test des imports")
    if not ok or "cuda True" not in out: mt_fail("test des imports (cuda doit être True)", out)
    print("   ", out.strip().splitlines()[-1])
    open(marker, "w").write("ok")

def mt_weights():
    HFR = "https://huggingface.co"; M = f"{MT_DIR}/models"
    dl = {
        f"{M}/musetalkV15/unet.pth":               f"{HFR}/TMElyralab/MuseTalk/resolve/main/musetalkV15/unet.pth",
        f"{M}/musetalkV15/musetalk.json":          f"{HFR}/TMElyralab/MuseTalk/resolve/main/musetalkV15/musetalk.json",
        f"{M}/sd-vae/config.json":                 f"{HFR}/stabilityai/sd-vae-ft-mse/resolve/main/config.json",
        f"{M}/sd-vae/diffusion_pytorch_model.bin": f"{HFR}/stabilityai/sd-vae-ft-mse/resolve/main/diffusion_pytorch_model.bin",
        f"{M}/whisper/config.json":                f"{HFR}/openai/whisper-tiny/resolve/main/config.json",
        f"{M}/whisper/pytorch_model.bin":          f"{HFR}/openai/whisper-tiny/resolve/main/pytorch_model.bin",
        f"{M}/whisper/preprocessor_config.json":   f"{HFR}/openai/whisper-tiny/resolve/main/preprocessor_config.json",
        f"{M}/dwpose/dw-ll_ucoco_384.pth":         f"{HFR}/yzd-v/DWPose/resolve/main/dw-ll_ucoco_384.pth",
        f"{M}/face-parse-bisent/79999_iter.pth":   f"{HFR}/ManyOtherFunctions/face-parse-bisent/resolve/main/79999_iter.pth",
        f"{M}/face-parse-bisent/resnet18-5c106cde.pth": "https://download.pytorch.org/models/resnet18-5c106cde.pth",
        f"{MT_DIR}/musetalk/utils/face_detection/detection/sfd/s3fd.pth": f"{HFR}/camenduru/Wav2Lip/resolve/main/face_detection/detection/sfd/s3fd.pth",
    }
    for path, url in dl.items():
        if os.path.exists(path) and os.path.getsize(path) > 1000: continue
        os.makedirs(os.path.dirname(path), exist_ok=True)
        ok, out = mt_run(f'wget -q -O "{path}.part" "{url}" && mv "{path}.part" "{path}"', "téléchargement " + os.path.basename(path))
        if not ok: mt_fail("téléchargement " + url, out)
    print("   ✅ tous les modèles sont là")

def mt_patch_upstream():
    """Le script officiel saute les images sans visage détecté, ce qui crée des trous dans la séquence.
    On écrit l'image d'origine à la place. Sans danger si relancé."""
    p = f"{MT_DIR}/scripts/inference.py"
    s = open(p).read()
    if "LIPSYNC_COLAB_PATCH" in s: return
    old = """                try:
                    res_frame = cv2.resize(res_frame.astype(np.uint8), (x2-x1, y2-y1))
                except:
                    continue
"""
    new = """                try:  # LIPSYNC_COLAB_PATCH: keep the PNG sequence gap-free
                    if bbox == coord_placeholder or x2 <= x1 or y2 <= y1: raise ValueError("no face")
                    res_frame = cv2.resize(res_frame.astype(np.uint8), (x2-x1, y2-y1))
                except Exception:
                    cv2.imwrite(f"{result_img_save_path}/{str(i).zfill(8)}.png", ori_frame)
                    continue
"""
    if old not in s:
        print("   ⚠️ le script officiel a changé ; correctif non appliqué"); return
    open(p, "w").write(s.replace(old, new)); print("   ▶ correctif appliqué (séquence d'images sans trou)")

def mt_find_output(out_dir, expected):
    if os.path.exists(expected) and os.path.getsize(expected) > 1000: return expected
    c = [f for f in glob.glob(f"{out_dir}/**/*.mp4", recursive=True)
         if not os.path.basename(f).startswith("temp_") and not f.endswith("_concat.mp4") and os.path.getsize(f) > 1000]
    return max(c, key=os.path.getmtime) if c else None

def mt_infer(video, audio):
    out_dir = "/content/work/mt"; out_mp4 = f"{out_dir}/v15/musetalk_raw.mp4"
    subprocess.run(f"rm -rf {out_dir}", shell=True)
    with open("/content/work/mt.yaml", "w") as f:
        f.write(f'task_0:\n  video_path: "{video}"\n  audio_path: "{audio}"\n')
    cmd = (f"cd {MT_DIR} && PYTHONUNBUFFERED=1 {PY} -m scripts.inference "
           f"--inference_config /content/work/mt.yaml --result_dir {out_dir} "
           f"--unet_model_path ./models/musetalkV15/unet.pth --unet_config ./models/musetalkV15/musetalk.json "
           f"--version v15 --whisper_dir ./models/whisper --vae_type sd-vae "
           f"--ffmpeg_path /usr/bin --fps {FPS} --batch_size 8 --bbox_shift 0 --use_float16 --output_vid_name musetalk_raw.mp4")
    mt_patch_upstream()
    ok, out = mt_run(cmd, "MuseTalk : synchronisation des lèvres")
    found = mt_find_output(out_dir, out_mp4)
    if not found:
        mt_fail("synchronisation (aucune vidéo produite ; si le journal dit « no face », utilise une photo de visage de face plus nette)", out)
    return found

try:
    print("Installation de MuseTalk...")
    mt_install()
    print("Téléchargement des modèles (environ 3,5 Go, ignoré s'ils sont déjà là)...")
    mt_weights()
    mt_patch_upstream()
    print("✅ MuseTalk est prêt. Passe à la cellule suivante (l'interface).")
except RuntimeError:
    raise SystemExit("❌ L'installation a échoué. Regarde le cadre d'erreur ci-dessus (ou /content/work/musetalk.log).")

In [ ]:
# ÉTAPE 3 : lance l'interface web (un lien public temporaire s'affiche)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "gradio", "edge-tts", "gTTS"])
import os, json, time
import gradio as gr
from PIL import Image, ImageOps

VID_EXT = (".mp4", ".mov", ".webm", ".mkv", ".avi")
VOICES = ["fr-FR-DeniseNeural", "fr-FR-HenriNeural", "fr-FR-EloiseNeural",
          "en-US-JennyNeural", "en-US-GuyNeural", "es-ES-ElviraNeural"]
MAX_SEC = 60

def sh(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError("Commande échouée : " + cmd[:120] + "\n" + (p.stderr or p.stdout)[-400:])
    return p.stdout

def generate(face_path, audio_path, text, voice, agree, progress=gr.Progress()):
    try:
        text = (text or "").strip()
        if not agree: raise RuntimeError("Coche la case d'accord avant de continuer.")
        if not face_path: raise RuntimeError("Ajoute une photo ou une vidéo de visage.")
        if not text and not audio_path: raise RuntimeError("Ajoute un audio (fichier ou micro), ou écris un texte.")

        # --- voix ---
        progress(0.05, desc="Préparation de la voix…")
        if text:
            open("/content/work/texte.txt", "w", encoding="utf-8").write(text)
            tts = "/content/work/voix_tts.mp3"
            if os.path.exists(tts): os.remove(tts)
            ok = subprocess.run(f"edge-tts --voice {voice} --file /content/work/texte.txt --write-media {tts}", shell=True).returncode == 0
            if not (ok and os.path.exists(tts) and os.path.getsize(tts) > 1000):
                from gtts import gTTS
                gTTS(text, lang=voice[:2]).save(tts)
            audio_src = tts
        else:
            audio_src = audio_path
        sh(f'ffmpeg -y -v error -i "{audio_src}" -ac 1 -ar 16000 /content/work/voice16k.wav')
        dur = float(sh("ffprobe -v error -show_entries format=duration -of csv=p=0 /content/work/voice16k.wav").strip())
        if dur > MAX_SEC: raise RuntimeError(f"Voix trop longue ({dur:.0f} s). Maximum {MAX_SEC} s.")

        # --- visage ---
        progress(0.15, desc="Préparation du visage…")
        if face_path.lower().endswith(VID_EXT):
            vf = f"scale='if(gt(iw,ih),min(1024,iw),-2)':'if(gt(iw,ih),-2,min(1024,ih))',fps={FPS},format=yuv420p"
            sh(f'ffmpeg -y -v error -stream_loop -1 -i "{face_path}" -an -t {dur + 0.2:.3f} -vf "{vf}" -c:v libx264 -crf 15 -preset fast /content/work/still.mp4')
            info = json.loads(sh("ffprobe -v error -select_streams v:0 -show_entries stream=width,height -of json /content/work/still.mp4"))["streams"][0]
            W, H = info["width"], info["height"]
        else:
            im = ImageOps.exif_transpose(Image.open(face_path)).convert("RGB")
            k = min(1, 1024 / max(im.size))
            im = im.resize((max(2, int(im.width * k) // 2 * 2), max(2, int(im.height * k) // 2 * 2)))
            im.save("/content/work/face.jpg", quality=95)
            W, H = im.size
            frames = int(round(dur * FPS)) + 2
            zs = 0.05 / max(frames, 1)
            vf = (f"scale={W*2}:{H*2}:flags=lanczos,zoompan=z='1+{zs:.8f}*on':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)'"
                  f":d={frames}:s={W}x{H}:fps={FPS},format=yuv420p")
            sh(f'ffmpeg -y -v error -i /content/work/face.jpg -vf "{vf}" -frames:v {frames} -c:v libx264 -crf 15 -preset fast /content/work/still.mp4')

        # --- synchronisation ---
        progress(0.3, desc="Synchronisation des lèvres (2 à 6 min)…")
        raw = mt_infer("/content/work/still.mp4", "/content/work/voice16k.wav")

        # --- finalisation ---
        progress(0.9, desc="Finalisation…")
        dst = f"/content/output/{OUTPUT_NAME}-{int(time.time())}.mp4"
        sh(f'ffmpeg -y -v error -i "{raw}" -i /content/work/voice16k.wav -map 0:v:0 -map 1:a:0 '
           f'-vf "scale={W}:{H}:flags=lanczos,format=yuv420p" -r {FPS} -c:v libx264 -preset medium -crf 17 '
           f'-c:a aac -b:a 192k -shortest -movflags +faststart "{dst}"')
        return dst
    except Exception as e:
        raise gr.Error(str(e)[:400])

with gr.Blocks(title="Visage parlant") as demo:
    gr.Markdown("# Visage parlant\nChoisis un visage et une voix : la photo ou la vidéo se met à parler.")
    face = gr.File(label="Photo ou vidéo du visage", file_types=["image", "video"], type="filepath")
    audio = gr.Audio(label="Ta voix (fichier ou micro)", sources=["upload", "microphone"], type="filepath")
    text = gr.Textbox(label="Ou écris un texte (prioritaire sur l'audio)", lines=3)
    voice = gr.Dropdown(VOICES, value=VOICES[0], label="Voix pour le texte")
    agree = gr.Checkbox(label="J'ai l'accord de la personne dont le visage et la voix sont utilisés.")
    btn = gr.Button("Animer", variant="primary")
    out = gr.Video(label="Résultat (indique « généré par IA » si tu publies)")
    btn.click(generate, [face, audio, text, voice, agree], out)

print("Ouvre le lien « public URL » ci-dessous (valable environ 72 h). Garde CET onglet Colab ouvert.")
demo.queue().launch(share=True, debug=True)